# 09 — Project summary and prediction

## Goal of this step

Review the complete workflow and its final evidence. Use the saved model and training-selected threshold to predict one genuine customer, then identify what remains unproven.

## Why this matters

A project is useful when another person can repeat it and understand its boundaries. The saved pipeline is like a recipe card: future predictions must use the exact same preparation as training.

Find the project root and import display tools. Reusable logic lives in src.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np  # noqa: F401
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.executable).resolve().is_relative_to(ROOT / ".venv"), "Select the project venv kernel."
%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4)


Read the final evidence rather than fitting or retesting. The table explicitly separates training-CV means from held-out test results.

In [2]:
from src.summary import results_table, run_summary
from src.predict import predict_one
from src.split import split_data
display(results_table())

,model,source,roc_auc,pr_auc_ap,precision,recall,f1
0,Dummy,Training 5-fold CV mean,0.500000,0.264423,0.000000,0.000000,0.000000
1,LogisticRegression,Training 5-fold CV mean,0.846514,0.660345,0.518234,0.803367,0.629980
2,XGBoost threshold=0.50,Held-out test,0.844012,0.659178,0.672535,0.513441,0.582317
3,XGBoost threshold=0.16,Held-out test,0.844012,0.659178,0.442077,0.892473,0.591273


**Interpretation:** The minority-class baseline is weak despite high accuracy; the final model ranks churners better, and its cost-selected threshold favors coverage over offer precision.

Demonstrate predict_one(dict) with a real training customer. Supply original predictor fields; the helper cleans, engineers, transforms with fitted statistics, and applies the saved threshold.

In [3]:
train, _, _, _ = split_data()
customer = train.iloc[0].to_dict()
probability, label = predict_one(customer)
display(pd.Series(customer, name='real_training_customer'))
display(pd.Series({'churn_probability': probability, 'offer_review_flag': label}))
run_summary()

gender                                 Male
SeniorCitizen                             0
Partner                                  No
Dependents                               No
tenure                                    2
PhoneService                            Yes
MultipleLines                            No
InternetService                 Fiber optic
OnlineSecurity                           No
OnlineBackup                             No
DeviceProtection                         No
TechSupport                              No
StreamingTV                              No
StreamingMovies                          No
Contract                     Month-to-month
PaperlessBilling                         No
PaymentMethod       Credit card (automatic)
MonthlyCharges                         70.7
TotalCharges                          140.7
Name: real_training_customer, dtype: object

churn_probability    0.596881
offer_review_flag    1.000000
dtype: float64

{'training_customer_row': 2624,
 'probability': 0.5968806147575378,
 'label': 1,
 'label_meaning': '1=eligible for a retention review, 0=no flag'}

**Interpretation:** The label is a threshold-based review flag, not certainty that the customer will leave or that an offer will help.

## Common mistakes

- Changing preprocessing at inference time.
- Treating one random split as proof of future performance.
- Claiming causal business benefits from churn prediction or SHAP.

## Key takeaways

- The pipeline and threshold provide reproducible inference.
- The holdout stayed outside feature/model/threshold decisions.
- A temporal validation and retention experiment are logical next steps.

## Next step

Review README.md, run the reproduction commands, and use the ten-step Git history to study each stage.

## What I learned

I learned that leakage can occur before fitting a model: test-driven EDA or threshold choice can also contaminate an evaluation. I kept learned preprocessing inside training folds and chose the threshold from training out-of-fold predictions.

I learned that imbalance makes accuracy a poor guide: a model that never finds churners can still score about 74%. I used average precision, recall, precision, and a dummy baseline to ask whether the model does something useful.

I learned that threshold choice is a business decision. A missed churner cost of five versus a wasted-offer cost of one favors recall, but those assumptions need real business validation.

I learned that interpretability explains a model, not causation. SHAP can highlight useful hypotheses while correlated features and the background sample affect how credit is divided.

## Limitations and next steps

This historical snapshot lacks a defined future prediction window, temporal test, retention outcomes, and deployment monitoring. Cleaning removes identical records after dropping IDs, a coursework simplification that may remove distinct customers. Engineered bill ratios are noisy proxies, and correlated fields share importance. The selected CV score and OOF threshold cost reuse training model-selection information. The XGBoost lead over logistic regression is small relative to fold variation, and tuning did not materially improve AP. Validate the label/prediction horizon, test a later time period, measure subgroup quality and calibration, and run a controlled retention experiment before operational use.